<a href="https://colab.research.google.com/github/medwicks/BAN482-MachineLearning/blob/main/BAN482_Supervised_Learning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# kNN- Classification

In [1]:
# Import needed packages for classification
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Import packages for visualization of results
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing
from mlxtend.plotting import plot_decision_regions

# Import packages for evaluation
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [12]:
# Load the bean dataset from the CSV file into a pandas DataFrame
beans = pd.read_csv('Dry_Bean_Dataset/MyDrive/SeniorFall/BAN482.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# Inspect a few rows
beans.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,41477,762.707,286.305731,184.828919,1.549031,0.763705,41886,229.804605,0.805285,0.990235,0.895987,0.802655,0.006903,0.001767,0.644254,0.997970,SIRA
1,39898,743.961,257.436106,197.919071,1.300714,0.639479,40460,225.387913,0.761528,0.986110,0.905859,0.875510,0.006452,0.002339,0.766518,0.997021,SEKER
2,45530,769.815,267.058152,217.385504,1.228500,0.580864,45833,240.770838,0.775745,0.993389,0.965461,0.901567,0.005866,0.002390,0.812823,0.998554,SEKER
3,43865,762.654,275.571786,202.806242,1.358793,0.677039,44217,236.327427,0.763507,0.992039,0.947704,0.857589,0.006282,0.002096,0.735459,0.999337,SEKER
4,74521,1037.103,405.065285,236.451948,1.713098,0.811942,75533,308.030979,0.784324,0.986602,0.870652,0.760448,0.005436,0.001121,0.578281,0.990651,CALI


In [14]:
# List the values in the Class column to check how many different labels exist
beans['Class'].unique()

array(['SIRA', 'SEKER', 'CALI', 'HOROZ', 'BARBUNYA', 'DERMASON', 'BOMBAY'],
      dtype=object)

In [15]:
# Standardize the bean class names by capitalizing the first letter only
beans['Class'] = beans['Class'].str.capitalize()

In [16]:
# Display the dimensions of the dataset (number of rows, number of columns)
beans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRation     13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  Compactness      13611 non-null  float64
 12  ShapeFactor1     13611 non-null  float64
 13  ShapeFactor2     13611 non-null  float64
 14  ShapeFactor3     13611 non-null  float64
 15  ShapeFactor4     13611 non-null  float64
 16  Class            13611 non-null  object 
dtypes: float64(1

In [17]:
# Generate descriptive statistics for all numerical columns in the dataset
beans.describe()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
count,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000
mean,53048.284549,855.283459,320.141867,202.270714,1.583242,0.750895,53768.200206,253.064220,0.749733,0.987143,0.873282,0.799864,0.006564,0.001716,0.643590,0.995063
std,29324.095717,214.289696,85.694186,44.970091,0.246678,0.092002,29774.915817,59.177120,0.049086,0.004660,0.059520,0.061713,0.001128,0.000596,0.098996,0.004366
min,20420.000000,524.736000,183.601165,122.512653,1.024868,0.218951,20684.000000,161.243764,0.555315,0.919246,0.489618,0.640577,0.002778,0.000564,0.410339,0.947687
25%,36328.000000,703.523500,253.303633,175.848170,1.432307,0.715928,36714.500000,215.068003,0.718634,0.985670,0.832096,0.762469,0.005900,0.001154,0.581359,0.993703
50%,44652.000000,794.941000,296.883367,192.431733,1.551124,0.764441,45178.000000,238.438026,0.759859,0.988283,0.883157,0.801277,0.006645,0.001694,0.642044,0.996386
75%,61332.000000,977.213000,376.495012,217.031741,1.707109,0.810466,62294.000000,279.446467,0.786851,0.990013,0.916869,0.834270,0.007271,0.002170,0.696006,0.997883
max,254616.000000,1985.370000,738.860154,460.198497,2.430306,0.911423,263261.000000,569.374358,0.866195,0.994677,0.990685,0.987303,0.010451,0.003665,0.974767,0.999733


### Setting up the Scikit Learn workflow

#### Define

In [18]:
# Select the two feature columns we'll use for classification: MajorAxisLength and MinorAxisLength
X = beans[['MajorAxisLength', 'MinorAxisLength']]
# Select the target variable (what we're trying to predict): the bean Class
y = beans[['Class']]

In [19]:
# Split the data into training (70%) and testing (30%) sets for model evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [20]:
# Create a StandardScaler object to standardize the features (rescale data to have mean=0 and variance=1)
scaler = StandardScaler()
# Fit the scaler to the data
# fit_transform calculates mean and standard deviation from the data, then applies the transformation to each value
# The output is a numpy array
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# create numpy array version of y_train to help with model fitting later
y_train_np = np.ravel(y_train)

#### Initialize

In [21]:
# Create a kNN classifier object with k=5 (will use 5 nearest neighbors for classification)
beanKnnClassifier = KNeighborsClassifier(n_neighbors=5)

# Note that k is a 'Hyperparameter'
# To keep it simple we are not going to optimize k for now
# That's the reason why we didnt do a three way split of the data

#### Fit

In [22]:
# Train the kNN model using the scaled training data
# np.ravel converts y_train from a 2D array to a 1D array (required by the fit method)
beanKnnClassifier.fit(X_train_scaled, y_train_np)

KNeighborsClassifier()

#### Predict

In [ ]:
# Make predictions on the test set using the trained model
# Transform X_test using the same scaler (with training data parameters) before predicting
y_pred = beanKnnClassifier.predict(X_test_scaled)

#### Evaluate

In [ ]:
# Calculate and print the accuracy score - the proportion of correct predictions
print(metrics.accuracy_score(y_test, y_pred))
# Generate and print the confusion matrix showing true vs predicted classifications for each bean class
print(metrics.confusion_matrix(y_test, y_pred))

### Model Application

Classifying a bean with MajorAxisLength = 400 and MinorAxisLength = 200

In [ ]:
# Create a DataFrame for a single bean to classify with MajorAxisLength=400 and MinorAxisLength=200
bean = pd.DataFrame(data={'MajorAxisLength': [400], 'MinorAxisLength': [200]})


# Use the trained model to predict the class of this single bean
# Scale the bean features using the same scaler before prediction
beanKnnClassifier.predict(scaler.transform(bean))

### Visualizing the regions of classification

In [ ]:
# Take a random sample of 200 beans to reduce computation time for visualization
# random_state=123 ensures reproducibility (same random sample each time)
beanSample = beans.sample(200, random_state=123)
# Display statistical summary of the sample data to verify it's representative
beanSample.describe()

In [ ]:
# Create a new column 'Int' with integer labels for each bean class (needed for visualization library)
# Replace each bean class name with a corresponding integer value (0-6)
beanSample['Int'] = beanSample['Class'].replace(
    to_replace = ['Barbunya', 'Bombay', 'Cali', 'Dermason', 'Horoz', 'Seker', 'Sira'],
    value = [int(0), int(1), int(2), int(3), int(4), int(5), int(6)])
# Extract the two feature columns (MajorAxisLength and MinorAxisLength) from the sample
X = beanSample[['MajorAxisLength', 'MinorAxisLength']]
# Extract the integer-encoded target labels
y = beanSample[['Int']]
# Train the kNN model on the sample data (without scaling for visualization purposes)
beanKnnClassifier.fit(X, np.ravel(y))

In [ ]:
# Create a dictionary to set the background opacity of decision regions to 20% (0.2) for better visibility
contourf_kwargs = {'alpha': 0.2}
# Plot the decision boundary regions showing how the kNN model classifies different areas
# X.to_numpy() converts the DataFrame to a NumPy array, np.ravel(y) flattens the target array
p = plot_decision_regions(X.to_numpy(), np.ravel(y), clf=beanKnnClassifier, contourf_kwargs=contourf_kwargs)
# Label the x-axis as 'MajorAxisLength' with font size 14 for readability
p.set_xlabel('MajorAxisLength', fontsize=14)
# Label the y-axis as 'MinorAxisLength' with font size 14 for readability
p.set_ylabel('MinorAxisLength', fontsize=14)
# Create a legend object for the plot
L = plt.legend()
# Update legend label for class 0 to display 'Barbunya'
L.get_texts()[0].set_text('Barbunya')
# Update legend label for class 1 to display 'Bombay'
L.get_texts()[1].set_text('Bombay')
# Update legend label for class 2 to display 'Cali'
L.get_texts()[2].set_text('Cali')
# Update legend label for class 3 to display 'Dermason'
L.get_texts()[3].set_text('Dermason')
# Update legend label for class 4 to display 'Horoz'
L.get_texts()[4].set_text('Horoz')
# Update legend label for class 5 to display 'Seker'
L.get_texts()[5].set_text('Seker')
# Update legend label for class 6 to display 'Sira'
L.get_texts()[6].set_text('Sira')

# kNN Regression

In [ ]:
# Import the KNeighborsRegressor class from sklearn for kNN regression modeling
from sklearn.neighbors import KNeighborsRegressor
# Import pandas library for data manipulation and analysis
import pandas as pd
# Import numpy library for numerical operations and array handling
import numpy as np
# Import StandardScaler for feature scaling/normalization
from sklearn.preprocessing import StandardScaler
# Import packages for visualization of results
# Import seaborn library for statistical data visualization
import seaborn as sns
# Import matplotlib.pyplot for creating plots and charts
import matplotlib.pyplot as plt
# Import packages for evaluation
# Import train_test_split function to divide data into training and testing sets
from sklearn.model_selection import train_test_split
# Import metrics module to evaluate model performance
from sklearn import metrics

In [ ]:
# Read data

beans = pd.read_csv('Dry_Bean_Dataset.csv')

### Setup the Scikit Learn workflow

#### Define

In [ ]:
# Define the Target and the input features
X = beans[['MajorAxisLength', 'MinorAxisLength']]
y = beans[['ConvexArea']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
# Scale Input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create numpy array version of y_train
y_train_numpy = np.ravel(y_train)

#### Initialize

In [ ]:
# Initialize model
beanKnnRegressor = KNeighborsRegressor(n_neighbors=5)

# k is a hyperparameter
# To keep it simple we are not going to optimize k for now
# That's the reason why we didnt do a three way split of the data

#### Fit

In [ ]:
# Fit model and predict on test data
beanKnnRegressor.fit(X_train_scaled, y_train_numpy)


#### Predict

In [ ]:
y_pred = beanKnnRegressor.predict(X_test_scaled)

#### Evaluate

In [ ]:
# Compute the mean squared error between the predicted and actual test values
print(metrics.mean_squared_error(y_pred, y_test))


In [ ]:
# Plot the actual value vs. the predicted value
plt.scatter(x=y_test, y=y_pred)

# Naive Bayes Classifier

In [ ]:
# Import packages and functions
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Read in the data and view the first five instances.
# File does not include column headers so they are provided via names.
messages = pd.read_table('SMSSpamCollection.csv', names=['Class', 'Message'])
messages.head()

In [ ]:
# Split into testing and training sets
X_train, X_test, Y_train, Y_test = train_test_split(
    messages['Message'], messages['Class'], random_state=123
)

In [ ]:
# Count the words that appear in the messages
vectorizer = CountVectorizer(ngram_range=(1, 1))
# Creates a CountVectorizer instance. ngram_range=(1, 1) means it will count
# individual words (unigrams).
# What CountVectorizer does
# CountVectorizer turns a list of text documents into numeric feature vectors by
# counting how often tokens appear. With a bag‑of‑words model, order is
# ignored; we only keep counts per token.
# Creating it: CountVectorizer(ngram_range=(1, 1)) sets it to extract only
# unigrams (single tokens). If you used (1, 2), it would include both unigrams
# and bigrams; (2, 2) would include only bigrams.​

# Fitting it: vectorizer.fit(X_train) scans all texts in X_train and builds a
# vocabulary (a mapping from token → column index). It does not transform data
# yet; it only learns which tokens exist and their indices.

# Transforming later: vectorizer.transform(X_train) (or fit_transform in one step)
# converts texts to a sparse matrix where rows are documents and columns are
# vocabulary tokens; cell values are counts.

vectorizer.fit(X_train)
# It learns the vocabulary of all unique words present in the training messages.

# Uncomment the line below to see the words.

# vectorizer.vocabulary_

# This attribute (when uncommented and displayed) shows the learned vocabulary
# as a dictionary where keys are the words and values are their corresponding
# index in the vocabulary. %% Count the words in the training set and store in a
# matrix

In [ ]:
# Count the words in the training set and store in a matrix
X_train_vectorized = vectorizer.transform(X_train)
X_train_vectorized

In [ ]:
# Initialize the model and fit with the training data
NBmodel = MultinomialNB()
NBmodel.fit(X_train_vectorized, Y_train)

In [ ]:
# Make predictions onto the training and testing sets.
trainPredictions = NBmodel.predict(vectorizer.transform(X_train))
testPredictions = NBmodel.predict(vectorizer.transform(X_test))

In [ ]:
# How does the model work on the training set?
confusion_matrix(Y_train, trainPredictions)

In [ ]:
# Display that in terms of correct proportions
confusion_matrix(Y_train, trainPredictions, normalize='true')

99.7% of real messages are classified correctly.
Just under 3% of spam messages are thought to be real.

In [ ]:
# How does the model work on the test set?
confusion_matrix(Y_test, testPredictions, normalize='true')

About 8.9% of spam messages are classified as real in the test data and about 0.7% of real messages are classified as spam.

In [ ]:
# Predict some phrases. Add your own.
NBmodel.predict(
    vectorizer.transform(
        ["Big sale today! Free cash.",
        "I'll be there in 5"]))